<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-02-grafos/18%20-%20Grafos%20de%20Logistica%20de%20Insumos%20e%20Produtos%20Acabados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<a href="https://colab.research.google.com/github/GuiCastro7/Grupo-3---ECAA08/blob/main/etapa-2-grafos/18%20-%20Grafos%20de%20Logistica%20de%20Insumos%20e%20Produtos%20Acabados.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Aula 18 - Notebook: Grafos de Logística de Insumos e Produtos Acabados

Neste notebook implementamos a modelagem em rede multicamada da fábrica de fertilizantes (`Grafo_Logitica.jpeg`): um Grafo Acíclico Dirigido (DAG) para o fluxo físico de materiais ($G_M$) e um dígrafo em anel para a circulação viária de veículos internos e rodoviários ($G_V$).

In [ ]:
from dataclasses import dataclass
from heapq import heappop, heappush
from math import inf
from typing import Dict, List, Tuple, Any

def formatar_tabela(dados: List[Dict[str, Any]]) -> str:
    """Formata lista de dicionarios em tabela ASCII pura."""
    if not dados:
        return "Tabela Vazia"
    colunas = list(dados[0].keys())
    larguras = {c: len(str(c)) for c in colunas}
    for row in dados:
        for c in colunas:
            larguras[c] = max(larguras[c], len(str(row.get(c, ""))))
    header = " | ".join(f"{c:<{larguras[c]}}" for c in colunas)
    divisor = "-+-".join("-" * larguras[c] for c in colunas)
    linhas = [header, divisor]
    for row in dados:
        linhas.append(" | ".join(f"{str(row.get(c, '')):<{larguras[c]}}" for c in colunas))
    return "\n".join(linhas)

@dataclass(frozen=True)
class Rota:
    destino: str
    peso: float
    descricao: str

class GrafoLogistico:
    def __init__(self) -> None:
        self._adjacencias: Dict[str, List[Rota]] = {}

    def adicionar_rota(self, origem: str, destino: str, peso: float, descricao: str) -> None:
        if peso < 0:
            raise ValueError("Dijkstra exige pesos não negativos.")
        self._adjacencias.setdefault(origem, []).append(Rota(destino, peso, descricao))
        self._adjacencias.setdefault(destino, [])

    def menor_caminho(self, origem: str, destino: str) -> Tuple[float, List[str]]:
        if origem not in self._adjacencias or destino not in self._adjacencias:
            raise KeyError("Origem e destino devem pertencer ao grafo.")

        distancias = {vertice: inf for vertice in self._adjacencias}
        predecessores: Dict[str, str] = {}
        distancias[origem] = 0.0
        fila: List[Tuple[float, str]] = [(0.0, origem)]

        while fila:
            distancia_atual, atual = heappop(fila)
            if distancia_atual != distancias[atual]:
                continue
            if atual == destino:
                break
            for rota in self._adjacencias[atual]:
                candidata = distancia_atual + rota.peso
                if candidata < distancias[rota.destino]:
                    distancias[rota.destino] = candidata
                    predecessores[rota.destino] = atual
                    heappush(fila, (candidata, rota.destino))

        if distancias[destino] == inf:
            raise ValueError(f"Não existe rota de {origem} para {destino}.")

        caminho = [destino]
        while caminho[-1] != origem:
            caminho.append(predecessores[caminho[-1]])
        caminho.reverse()
        return distancias[destino], caminho

def adicionar_rotas(grafo: GrafoLogistico, rotas: List[Tuple[str, str, float, str]]) -> None:
    for origem, destino, peso, descricao in rotas:
        grafo.adicionar_rota(origem, destino, peso, descricao)

# ------------------------------------------------------------------------------
# 1. GM: Grafo de Fluxo de Materiais (DAG - Directed Acyclic Graph)
# ------------------------------------------------------------------------------
fluxo_materiais = GrafoLogistico()
adicionar_rotas(fluxo_materiais, [
    ("Portaria", "Balança de entrada", 25, "liberação e pesagem do recebimento"),
    ("Balança de entrada", "Recebimento / Galpão A", 55, "direcionamento à descarga"),
    ("Recebimento / Galpão A", "Box: ureia", 18, "armazenagem de fonte nitrogenada"),
    ("Recebimento / Galpão A", "Box: fosfato", 20, "armazenagem de fonte fosfatada"),
    ("Recebimento / Galpão A", "Box: KCl", 22, "armazenagem de cloreto de potássio"),
    ("Recebimento / Galpão A", "Box: orgânicos e micronutrientes", 25, "armazenagem de complementos"),
    ("Recebimento / Galpão A", "Tanques líquidos", 28, "recebimento de aditivos líquidos"),
    ("Box: ureia", "Moega de recepção", 35, "transferência por correia"),
    ("Box: fosfato", "Moega de recepção", 32, "transferência por correia"),
    ("Box: KCl", "Moega de recepção", 30, "transferência por correia"),
    ("Box: orgânicos e micronutrientes", "Moega de recepção", 38, "transferência por correia"),
    ("Moega de recepção", "Silos de dosagem", 40, "alimentação dos dosadores"),
    ("Tanques líquidos", "Silos de dosagem", 45, "dosagem de aditivos"),
    ("Silos de dosagem", "Moinho / misturador", 18, "homogeneização da fórmula"),
    ("Moinho / misturador", "Granulação", 20, "formação dos grânulos"),
    ("Granulação", "Secador / classificação", 32, "secagem e peneiramento"),
    ("Secador / classificação", "Ensacamento / paletização", 48, "acabamento e embalagem"),
    ("Ensacamento / paletização", "Estoque NPK 04-14-08", 35, "endereçamento do produto acabado"),
    ("Ensacamento / paletização", "Estoque NPK 10-10-10", 42, "endereçamento do produto acabado"),
    ("Ensacamento / paletização", "Estoque Organomineral Premium", 50, "endereçamento do produto acabado"),
    ("Estoque NPK 04-14-08", "Docas de expedição", 28, "separação para carregamento"),
    ("Estoque NPK 10-10-10", "Docas de expedição", 25, "separação para carregamento"),
    ("Estoque Organomineral Premium", "Docas de expedição", 32, "separação para carregamento"),
])

# Cálculo da Rota Global de Referência (Portaria -> Docas)
dist_total, caminho_total = fluxo_materiais.menor_caminho("Portaria", "Docas de expedição")

print("=== FLUXO DE MATERIAIS: ROTA MÍNIMA GLOBAL DE PROCESSO (GM) ===")
print(" -> ".join(caminho_total))
print(f"Distância Total Estimada: {dist_total:.1f} m\n")

# Auditoria por Insumo Específico até as Docas de Expedição
relatorio_insumos = []
insumos_teste = [
    "Box: ureia",
    "Box: fosfato",
    "Box: KCl",
    "Box: orgânicos e micronutrientes",
    "Tanques líquidos"
]

for insumo in insumos_teste:
    dist_ins, rota_ins = fluxo_materiais.menor_caminho(insumo, "Docas de expedição")
    relatorio_insumos.append({
        "Insumo / Ponto de Origem": insumo,
        "Distância até Doca (m)": f"{dist_ins:.1f}",
        "Primeira Etapa Intermediária": rota_ins[1],
        "Rota Operacional Completa": " -> ".join(rota_ins)
    })

print("=== ROTAS DOS INSUMOS ATÉ A EXPEDIÇÃO FINAL ===")
print(formatar_tabela(relatorio_insumos))

# Validações de processo do DAG de materiais
assert "Silos de dosagem" in caminho_total, "Toda transformação deve passar pelos silos de dosagem!"
assert "Ensacamento / paletização" in caminho_total, "O produto deve passar por paletização antes da doca!"
assert dist_total == 334.0, f"Distância esperada de 334.0m, obtido: {dist_total}"
# ------------------------------------------------------------------------------
# 2. GV: Grafo de Circulação Viária de Veículos (Anel Viário Fechado)
# ------------------------------------------------------------------------------
circulacao_veiculos = GrafoLogistico()
anel_viario = [
    ("Portaria", "Balança de entrada", 25, "entrada de veículos"),
    ("Balança de entrada", "Pátio de recebimento", 55, "acesso à descarga"),
    ("Pátio de recebimento", "Galpão A", 40, "rota de descarga"),
    ("Galpão A", "Área de processamento", 65, "rota interna de serviço"),
    ("Área de processamento", "Galpão B", 75, "rota para armazenagem"),
    ("Galpão B", "Docas de expedição", 30, "rota de carregamento"),
    ("Docas de expedição", "Balança de saída", 60, "saída de veículos"),
    ("Balança de saída", "Portaria", 35, "liberação de saída"),
]
adicionar_rotas(circulacao_veiculos, anel_viario)

# Rota de Saída do Caminhão após Carga
dist_saida, rota_saida = circulacao_veiculos.menor_caminho("Docas de expedição", "Portaria")

print("\n=== CIRCULAÇÃO DE VEÍCULOS APÓS O CARREGAMENTO (GV) ===")
print(" -> ".join(rota_saida))
print(f"Distância do Pátio de Saída: {dist_saida:.1f} m")

# Asserções formais da circulação rodoviária
assert rota_saida == ["Docas de expedição", "Balança de saída", "Portaria"]
assert dist_saida == 95.0, f"Distância esperada de 95.0m, obtido: {dist_saida}"
print("\n✓ Todos os modelos logísticos (GM e GV) foram validados com 100% de sucesso!")